# Laya 1.2 — **FORCED ALL 28 layers** | mixed 2/4/8-bit G128 + accuracy recovery

**Source model:** [vtava/Laya-1.2-speedup-standalone-fp16](https://huggingface.co/vtava/Laya-1.2-speedup-standalone-fp16)  
**GitHub:** [TinyCeNN-LM](https://github.com/vtavakkoli/TinyCeNN-LM)  
**Goal:** Compress *every eligible projection in every encoder layer* while retaining as many original FP16 decisions as possible.

## Why this replaces V2

V2 on Tesla T4 selected **4 of 24 projections in layers 0–5**, reduced packed state **803.05 → 781.80 MiB (2.65%)**, reached **97.40% agreement on dev**, **95.31% on independent audit**, and **96.5% on official test**. The model used PyTorch SDPA and FP16 cached GEMM, giving close to baseline latency but **806.3 MiB resident weights**. V2's activation-aware ridge fitting improved layer-output MSE but was *not enough* for decision fidelity.

**V3 is intentionally a different experiment:** instead of forcing every weight to 2-bit ternary and watching accuracy collapse, it **forcefully replaces all 4 linear projections in each of the 28 encoder blocks** with a choice of **2-bit ternary**, **4-bit symmetric**, and **8-bit symmetric** groupwise weights.

For each layer, test predeclared precision profiles; commit the smallest profile meeting cumulative development fidelity; if nothing passes, **force 8-bit rather than reverting to FP16**. After all layers, run global **8-bit precision recovery** if needed, followed by an independent audit and untouched official test. This is NOT guaranteed to improve the model's *absolute task accuracy* compared with the original; it aims to preserve it while reducing packed checkpoint size.

- All linear projections are stored in quantized 2/4/8-bit format (FP16 scale/G128). None is deliberately left FP16.
- ModernBERT attention uses **PyTorch SDPA**; projections use native **FP16 PyTorch `F.linear`** after cached dequantization, which retains conventional GEMM speed but **does not retain low-bit weight residency in GPU VRAM**.
- Compact checkpoint and true resident GPU memory are reported separately; claimed speedups are measured, never assumed.
- No training on the official test. Source HF model remains unchanged.
- Typical T4/L4 run may require substantial time for all-layer candidate evaluation; per-layer JSON/CSV is checkpointed.

**Safety rule:** A failed fidelity gate is reported even if all modules were forcibly replaced. No automatic publication of an unvalidated compressed model.


In [ ]:
#@title 1. Install dependencies (native PyTorch SDPA; no custom Triton requirement)
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4" "datasets>=2.20,<5" pandas matplotlib


In [ ]:
#@title 2. Configure and download original FP16 standalone model
import sys, os, gc, json, random, math, time, zipfile, inspect, importlib.util
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F
from huggingface_hub import snapshot_download
from IPython.display import display

SOURCE_REPO="vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
TRAIN_CASES=1000 #@param {type:"integer"}
DEV_DECISIONS=256 #@param {type:"integer"}
AUDIT_DECISIONS=256 #@param {type:"integer"}
OFFICIAL_TEST_DECISIONS=400 #@param {type:"integer"}
MAX_LEN=512 #@param {type:"integer"}
BATCH_SIZE=4 #@param {type:"integer"}
MIN_DEV_AGREEMENT=0.99 #@param {type:"number"}
MIN_AUDIT_AGREEMENT=0.99 #@param {type:"number"}
MAX_MEAN_JS=0.004 #@param {type:"number"}
BENCH_REPEATS=8 #@param {type:"integer"}
MIN_LINEAR_ELEMENTS=4096 #@param {type:"integer"}
ENABLE_GLOBAL_PRECISION_RESCUE=True #@param {type:"boolean"}
SAVE_PACKED_DELTA=True #@param {type:"boolean"}
SEED=2033
OUT=Path("/content/Laya_Force_All28_MixedBit")
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select T4 / L4 GPU in Colab Runtime settings."
assert torch.cuda.get_device_capability()[0]>=7, "Tensor-core GPU required."
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN=os.environ.get("HF_TOKEN")
ROOT=Path(snapshot_download(repo_id=SOURCE_REPO,token=HF_TOKEN))
CFG=json.loads((ROOT/"config.json").read_text())
assert CFG.get("format")=="laya-1p2-full-standalone-v1"
assert CFG.get("dtype_on_disk")=="float16"
print("GPU:",torch.cuda.get_device_name(0),"| input length:",MAX_LEN)
print("Loaded original checkpoint files:",len(list(ROOT.iterdir())))


In [ ]:
#@title 3. Load the exact LayaStandalone architecture and request PyTorch SDPA
def import_module(file,name):
    spec=importlib.util.spec_from_file_location(name,str(file))
    if spec is None or spec.loader is None:raise ImportError(str(file))
    m=importlib.util.module_from_spec(spec)
    sys.modules[name]=m
    spec.loader.exec_module(m)
    return m

source=import_module(ROOT/"modeling_laya_standalone.py","_laya_standalone_forced_v3")
teacher,tokenizer=source.load_model(ROOT,device="cpu",
                                  local_files_only=True,compile_head=False)
teacher=teacher.to("cuda",dtype=torch.float16).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
try:
    if hasattr(teacher.encoder,"set_attn_implementation"):
        teacher.encoder.set_attn_implementation("sdpa")
    else:
        teacher.encoder.config._attn_implementation="sdpa"
except Exception as error:
    print("Original SDPA path retained:",str(error)[:180])
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
assert hasattr(teacher.encoder,"layers")
N_LAYERS=len(teacher.encoder.layers)
print("Discovered",N_LAYERS,"encoder layers")
assert N_LAYERS==28, f"This forced-all-28 notebook expects exactly 28; got {N_LAYERS}"
print("Original parameter count:",sum(p.numel() for p in teacher.parameters()))


In [ ]:
#@title 4. Fetch pinned 2/4/8-bit native PyTorch packed-weight runtime; smoke test on CPU and GPU
from urllib.request import urlopen
PIN="c6cc6cc250831e7f1a1c6688832c2e8a427355ac"
NAME="laya_forced_mixed_precision.py"
URL=f"https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/{PIN}/src/tinycenn_lm/{NAME}"
local=Path("/content")/NAME
with urlopen(URL,timeout=90) as response:local.write_bytes(response.read())
assert local.stat().st_size>5000, "Runtime download incomplete"
q=import_module(local,"laya_forced_mixed_precision")

# Check exact code packing and dequantization for all three bit widths.
for bits in (2,4,8):
    test_linear=nn.Linear(143,93,device="cuda",dtype=torch.float16)
    packed=q.PackedMixedLinear.from_linear(test_linear,bits)
    roundtrip=q.unpack_codes(packed.packed_weight,bits,packed.padded)
    assert torch.equal(q.pack_codes(roundtrip,bits),packed.packed_weight)
    xx=torch.randn(16,143,device="cuda",dtype=torch.float16)
    with torch.inference_mode():
        expected=F.linear(xx,packed.weight_fp16(),packed.bias)
        actual=packed(xx)
    torch.testing.assert_close(actual,expected,atol=0.01,rtol=0.01)
    print(f"{bits}-bit: PASS | packed MiB={q.state_bytes(packed)/2**20:.5f}"
          f" | resident FP16 cache MiB={packed.cache_bytes()/2**20:.5f}")
    del test_linear,packed,xx
gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 5. Build disjoint real typed-decision DEV / AUDIT data from the train split
from datasets import load_dataset
def parse_json(value):
    if isinstance(value,str):
        try:return json.loads(value)
        except Exception:return value
    return value

def prepare_items(rows,target):
    groups=[]
    for row in rows:
        state=parse_json(row.get("state"))
        questions=parse_json(row.get("questions"))
        if not isinstance(questions,dict):continue
        case=[]
        for question in questions.values():
            try:
                normalized=source.normalize_question(parse_json(question))
                ids,markers=source.build_sequence(
                    tokenizer,state,normalized,max_len=MAX_LEN,
                    head_max_len=192,truncate_left=isinstance(state,list))
                if len(markers)<2:continue
                case.append(dict(ids=ids,markers=markers,qtype=source.QTYPES[normalized["t"]]))
            except (TypeError,ValueError,KeyError,IndexError):
                continue
        if case:groups.append(case)
    # Round-robin cases instead of grabbing many questions from a few cases.
    result=[]
    for k in range(max(map(len,groups),default=0)):
        for case in groups:
            if k<len(case):
                result.append(case[k])
                if len(result)>=target:return result
    return result

rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split=f"train[:{TRAIN_CASES}]"))
random.Random(SEED).shuffle(rows)
mid=len(rows)//2
DEV_ITEMS=prepare_items(rows[:mid],DEV_DECISIONS)
AUDIT_ITEMS=prepare_items(rows[mid:],AUDIT_DECISIONS)
if len(DEV_ITEMS)<DEV_DECISIONS or len(AUDIT_ITEMS)<AUDIT_DECISIONS:
    raise RuntimeError(f"Not enough disjoint real decisions: dev={len(DEV_ITEMS)}, audit={len(AUDIT_ITEMS)}")

def batchify(items,size=BATCH_SIZE,pad_len=None):
    batches=[]
    for offset in range(0,len(items),size):
        b=source.collate_items(items[offset:offset+size],int(tokenizer.pad_token_id))
        if pad_len is not None:
            missing=pad_len-b["input_ids"].shape[1]
            if missing<0:raise ValueError(f"Sequence too long for fixed bench length {pad_len}")
            b["input_ids"]=F.pad(b["input_ids"],(0,missing),value=int(tokenizer.pad_token_id))
            b["attention_mask"]=F.pad(b["attention_mask"],(0,missing))
        batches.append({key:t.cuda() for key,t in b.items()})
    return batches

DEV=batchify(DEV_ITEMS)
AUDIT=batchify(AUDIT_ITEMS)
print("Disjoint decisions",len(DEV_ITEMS),"development,",len(AUDIT_ITEMS),"audit")


In [ ]:
#@title 6. Capture teacher logits and compute stable agreement / JS for real options
@torch.inference_mode()
def infer(model,b):
    with torch.autocast("cuda",dtype=torch.float16):
        z=model(**b)
    return z[0] if isinstance(z,(tuple,list)) else z

@torch.inference_mode()
def capture(model,batches):
    return [infer(model,b).detach().float().cpu() for b in batches]
DEV_TEACHER=capture(teacher,DEV)
AUDIT_TEACHER=capture(teacher,AUDIT)

@torch.inference_mode()
def compare(model,batches,truth):
    count=hits=0;total_js=0.0
    max_l1=0.0
    for b,logits in zip(batches,truth):
        pred=infer(model,b).detach().float().cpu()
        mask=b["marker_mask"].cpu()
        pred=pred.masked_fill(~mask,-1e4)
        ref=logits.masked_fill(~mask,-1e4)
        probs=pred.softmax(-1).clamp_min(1e-9)
        target=ref.softmax(-1).clamp_min(1e-9)
        mix=0.5*(probs+target)
        js=0.5*(probs*(probs/mix).log()).sum(-1)
        js+=0.5*(target*(target/mix).log()).sum(-1)
        count+=len(pred)
        hits+=int((probs.argmax(-1)==target.argmax(-1)).sum())
        total_js+=float(js.sum())
        max_l1=max(max_l1,float((pred-ref).abs().max()))
    return dict(n=count,agreement=hits/count,mean_js=total_js/count,max_logit_abs=max_l1)

def quality_ok(m):
    return m["agreement"]>=MIN_DEV_AGREEMENT and m["mean_js"]<=MAX_MEAN_JS
print("Teacher self-check:",compare(teacher,DEV,DEV_TEACHER))


In [ ]:
#@title 7. Full 28-layer forced mixed-precision PTQ: no FP16 rollback
import copy
student=copy.deepcopy(teacher).eval().requires_grad_(False)
BASE_BYTES=q.state_bytes(teacher)

def available_projection_names(m,layer):
    prefix=f"encoder.layers.{layer}."
    return [(name,sub) for name,sub in m.named_modules()
            if name.startswith(prefix) and isinstance(sub,nn.Linear)
            and sub.weight.numel()>=MIN_LINEAR_ELEMENTS]

def profile_bit(name,profile):
    suffix=name.rsplit(".",2)[-2:]
    kind,last=suffix
    if profile=="ternary_all":return 2
    if profile=="int4_all":return 4
    if profile=="int8_all":return 8
    if profile=="mixed_small":
        return 8 if kind=="attn" and last=="Wqkv" else 2 if kind=="attn" else 4
    if profile=="mixed_safe":
        return 8 if (kind=="attn" and last=="Wqkv") or (kind=="mlp" and last=="Wo") else 4
    raise ValueError(profile)

# Smallest first; a profile is accepted only if its cumulative quality gate passes.
PROFILES=("ternary_all","mixed_small","int4_all","mixed_safe","int8_all")
results=[];selected_bits={};layer_summary=[]
for index in range(N_LAYERS):
    original=available_projection_names(student,index)
    if len(original)!=4:
        raise RuntimeError(f"Expected 4 eligible projection matrices in layer {index}, got {[name for name,_ in original]}")
    names=[name for name,_ in original]
    variants={name:{bits:q.PackedMixedLinear.from_linear(module,bits)
                     for bits in (2,4,8)}
              for name,module in original}
    original_modules=dict(original)
    attempts=[];best=None
    for profile in PROFILES:
        mapping={name:profile_bit(name,profile) for name in names}
        for name in names:q.swap(student,name,variants[name][mapping[name]])
        try:
            measure=compare(student,DEV,DEV_TEACHER)
            storage=sum(q.state_bytes(variants[name][mapping[name]]) for name in names)
            passed=quality_ok(measure)
            record=dict(layer=index,profile=profile,passed=bool(passed),
                        local_packed_mib=storage/2**20,**measure)
            results.append(record)
            attempts.append((storage,measure["mean_js"],profile,mapping,measure))
            print(f"L{index:02d} {profile:13s} agreement {measure['agreement']:.3%}"
                  f" JS {measure['mean_js']:.5f} {'PASS' if passed else 'FAIL'}")
        except Exception as exc:
            print(f"L{index:02d} {profile} FAILED:",repr(exc)[:200])
            results.append(dict(layer=index,profile=profile,passed=False,error=repr(exc)))
        finally:
            for name in names:q.swap(student,name,original_modules[name])

    passes=[rec for rec in attempts if quality_ok(rec[4])]
    if passes:
        best=min(passes,key=lambda rec:(rec[0],rec[1]))
        forced=False
    else:
        fallback=[rec for rec in attempts if rec[2]=="int8_all"]
        if not fallback:raise RuntimeError(f"Layer {index}: even int8 profile crashed — do not silently keep FP16.")
        best=fallback[0]
        forced=True
    _,_,chosen,mapping,measure=best
    for name in names:
        q.swap(student,name,variants[name][mapping[name]])
        selected_bits[name]=mapping[name]
    record=dict(layer=index,choice=chosen,forced_int8=bool(forced),
                converted_count=len(selected_bits),
                packed_state_mib=q.state_bytes(student)/2**20,**measure)
    layer_summary.append(record)
    print(f"  -> L{index:02d} FORCED CONVERT: {chosen}, "
          f"{len(selected_bits)} / {4*N_LAYERS} projections, "
          f"agreement={measure['agreement']:.3%}, packed state={record['packed_state_mib']:.1f} MiB")
    pd.DataFrame(results).to_csv(OUT/"all_layer_profile_trials.csv",index=False)
    pd.DataFrame(layer_summary).to_csv(OUT/"all_layer_progress.csv",index=False)
    (OUT/"selected_precision.json").write_text(json.dumps(selected_bits,indent=2))
    del variants, original, original_modules, attempts
    gc.collect();torch.cuda.empty_cache()

assert len(selected_bits)==4*N_LAYERS, f"All-layer force guarantee violated: {len(selected_bits)}"
assert all(not available_projection_names(student,layer) for layer in range(N_LAYERS))
print(f"ALL {N_LAYERS} LAYERS FORCIBLY MODIFIED / {len(selected_bits)} PACKED PROJECTIONS")
display(pd.DataFrame(layer_summary))


In [ ]:
#@title 8. Global accuracy rescue: upgrade sensitive projections to int8 but NEVER FP16
PRE_RESCUE=compare(student,DEV,DEV_TEACHER)
print("Before recovery:",PRE_RESCUE)
rescue_rows=[]
if ENABLE_GLOBAL_PRECISION_RESCUE and not quality_ok(PRE_RESCUE):
    rescue_order=[
        [name for name in selected_bits if ".attn.Wqkv" in name],
        [name for name in selected_bits if ".mlp.Wo" in name],
        [name for name in selected_bits if ".mlp.Wi" in name],
        [name for name in selected_bits if ".attn.Wo" in name],
    ]
    for stage,names in enumerate(rescue_order):
        count=0
        for name in names:
            if selected_bits[name]==8:continue
            fresh=q.PackedMixedLinear.from_linear(teacher.get_submodule(name),8)
            q.swap(student,name,fresh)
            selected_bits[name]=8
            count+=1
        score=compare(student,DEV,DEV_TEACHER)
        rec=dict(stage=stage,upgraded_to_int8=count,**score,
                 packed_state_mib=q.state_bytes(student)/2**20)
        rescue_rows.append(rec)
        print(f"Precision rescue {stage} (+{count} int8): "
              f"agreement={score['agreement']:.3%} JS={score['mean_js']:.6f}")
        pd.DataFrame(rescue_rows).to_csv(OUT/"precision_rescue.csv",index=False)
        (OUT/"selected_precision.json").write_text(json.dumps(selected_bits,indent=2))
        if quality_ok(score):break
FINAL_DEV=compare(student,DEV,DEV_TEACHER)
print("Final selection DEV:",FINAL_DEV)
print("Quantized modules after global rescue:",len(selected_bits),
      "| bit counts:",{bit:list(selected_bits.values()).count(bit) for bit in (2,4,8)})
assert len(selected_bits)==4*N_LAYERS


In [ ]:
#@title 9. Independent holdout and OFFICIAL test — only after precision selection is frozen
FINAL_AUDIT=compare(student,AUDIT,AUDIT_TEACHER)
print("Independent training-case audit:",FINAL_AUDIT)
assert FINAL_AUDIT["n"]==AUDIT_DECISIONS
TEST_REPORT=None
if OFFICIAL_TEST_DECISIONS>0:
    test_rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="test"))
    test_items=prepare_items(test_rows,OFFICIAL_TEST_DECISIONS)
    if len(test_items)<OFFICIAL_TEST_DECISIONS:
        print(f"Only {len(test_items)} valid official decisions available")
    if test_items:
        official_batch=batchify(test_items)
        test_logits=capture(teacher,official_batch)
        TEST_REPORT=compare(student,official_batch,test_logits)
        print("Official TEST — never used for quantization selection:",TEST_REPORT)
AUDIT_PASS=(FINAL_AUDIT["agreement"]>=MIN_AUDIT_AGREEMENT
            and FINAL_AUDIT["mean_js"]<=MAX_MEAN_JS)
print("Independent audit:", "PASS" if AUDIT_PASS else "FAIL")


In [ ]:
#@title 10. Benchmark whole-model PyTorch SDPA + cached FP16 GEMM (true latency)
@torch.inference_mode()
def timer(fn,repeats=BENCH_REPEATS,warmup=3):
    for _ in range(warmup):fn()
    torch.cuda.synchronize()
    values=[]
    for _ in range(repeats):
        start=torch.cuda.Event(enable_timing=True)
        finish=torch.cuda.Event(enable_timing=True)
        start.record();fn();finish.record()
        torch.cuda.synchronize()
        values.append(start.elapsed_time(finish))
    return float(np.median(values))

bench_rows=[]
for batch_size in (1,4):
    samples=DEV_ITEMS[:batch_size]
    shortest=max(len(x["ids"]) for x in samples)
    for t in (128,256,512):
        if t<shortest or t>MAX_LEN:continue
        b=batchify(samples,size=batch_size,pad_len=t)[0]
        # Warm both model kernels and *student* reconstructed FP16 caches.
        baseline=timer(lambda:infer(teacher,b))
        converted=timer(lambda:infer(student,b))
        bench_rows.append(dict(batch=batch_size,tokens=t,
                              original_fp16_ms=baseline,mixed_bit_native_ms=converted,
                              speedup=baseline/converted,
                              packed_state_mib=q.state_bytes(student)/2**20,
                              resident_state_mib=q.live_bytes(student)/2**20))
        print(f"B{batch_size} T{t}: FP16 {baseline:.2f} ms,"
              f" packed+FP16 {converted:.2f} ms,"
              f" speedup {baseline/converted:.3f}x")
bench=pd.DataFrame(bench_rows)
assert not bench.empty
bench.to_csv(OUT/"inference_latency.csv",index=False)
display(bench)


In [ ]:
#@title 11. Summarize actual compression, quality, forced coverage, and speed
from collections import Counter
mix=Counter(selected_bits.values())
report={
    "model_repo":SOURCE_REPO,
    "gpu":torch.cuda.get_device_name(0),
    "architecture":"LayaStandalone ModernBERT + PyTorch SDPA",
    "conversion_policy":"FORCE all 28 encoder layers and all 4 eligible linear projections",
    "quantization":"symmetric G128 per-output, signed ternary2/int4/int8; FP16 group scale",
    "bits_count":dict(mix),
    "encoder_layers_modified":N_LAYERS,
    "linear_projections_quantized":len(selected_bits),
    "original_state_mib":q.state_bytes(teacher)/2**20,
    "packed_state_mib":q.state_bytes(student)/2**20,
    "packed_reduction_percent":100*(1-q.state_bytes(student)/q.state_bytes(teacher)),
    "live_cached_fp16_mib":q.live_bytes(student)/2**20,
    "dev":FINAL_DEV,"audit":FINAL_AUDIT,"official_test":TEST_REPORT,
    "dev_pass":quality_ok(FINAL_DEV),
    "audit_pass":bool(AUDIT_PASS),
    "ready_for_production":False,
    "notices":[
        "No original FP16 projection is preserved among the 112 encoder projections",
        "FP16 reconstructed caches occupy resident GPU memory; packed disk compression does not imply VRAM compression",
        "This is post-training mixed precision, not QAT or proof of higher absolute task accuracy",
        "Official test was evaluated only after all precision choices were frozen",
        "The source model is not overwritten"]
}
(OUT/"summary_forced_all28.json").write_text(json.dumps(report,indent=2))
print(json.dumps({k:report[k] for k in
 ("bits_count","encoder_layers_modified","linear_projections_quantized",
 "original_state_mib","packed_state_mib","packed_reduction_percent",
 "live_cached_fp16_mib","dev","audit","official_test","audit_pass")},indent=2))


In [ ]:
#@title 12. Save experimental PACKED weight delta, with per-projection bit metadata
from safetensors.torch import save_file
(OUT/"selected_precision.json").write_text(json.dumps(selected_bits,indent=2))
if SAVE_PACKED_DELTA:
    state={}
    for fullname,bits in selected_bits.items():
        module=student.get_submodule(fullname)
        if not isinstance(module,q.PackedMixedLinear) or module.bits!=bits:
            raise RuntimeError(f"Invalid packed module {fullname}")
        for suffix,tensor in module.state_dict().items():
            state[fullname+"."+suffix]=tensor.detach().contiguous().cpu()
    destination=OUT/"laya_forced_all28_mixedbit_delta.safetensors"
    save_file(state,str(destination))
    (OUT/"delta_manifest.json").write_text(json.dumps({
        "source_model":SOURCE_REPO,
        "required_base":"the exact FP16 standalone model revision",
        "packed_module_bits":selected_bits,
        "group_size":128,
        "format":"symmetric signed 2/4/8 bit with FP16 output-channel G128 group scale",
        "load_note":"Experiment is a delta, NOT an independently runnable Hugging Face model",
        "quality_warning":"Inspect final audit and official test before deploying"
    },indent=2))
    print("Packed module delta stored:",destination,
          "MiB",round(destination.stat().st_size/2**20,2))


In [ ]:
#@title 13. Show coverage and output accuracy vs size over 28 layers
import matplotlib.pyplot as plt
progress=pd.DataFrame(layer_summary)
fig,axes=plt.subplots(1,2,figsize=(13,4.2))
axes[0].plot(progress.layer,progress.agreement,marker=".")
axes[0].axhline(MIN_DEV_AGREEMENT,linestyle="--",label="Dev agreement threshold")
axes[0].set(xlabel="Encoder layer",ylabel="Agreement to FP16",
            title="Forced conversion: cumulative fidelity",ylim=(0,1.01))
axes[0].legend()
axes[1].plot(progress.layer,progress.packed_state_mib,marker=".")
axes[1].axhline(BASE_BYTES/2**20,linestyle="--",label="Original FP16")
axes[1].set(xlabel="Encoder layer",ylabel="Packed state MiB",
            title="Compressed checkpoint bytes")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT/"all28_fidelity_size.png",dpi=175)
plt.show()


In [ ]:
#@title 14. Download all evidence, numerical results and optional packed model delta
archive=OUT/"Laya_Force_All28_MixedBit_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p!=archive:
            z.write(p,p.name)
print("Artifact:",archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass


## Interpreting V3

**Coverage:** The code asserts **28 of 28 encoder layers and 112 of 112 eligible FP16 linear projections are forcibly changed** into 2-bit, 4-bit, or 8-bit packed modules. Mixed precision is allowed because the previous results showed full-ternary quantization degrades final decisions.

**Quality:** Every profile is compared with the original FP16 teacher on development data. If no profile meets the gate, 8-bit is forced rather than returning to FP16. After all layers, any failing global fidelity can trigger a cascade of 8-bit upgrades. Official test is only evaluated at the end and **must not guide a second search over the same test cases**.

**Performance:** Inference uses native PyTorch FP16 cached weights and SDPA attention. Packed weights reduce checkpoint size, but decoded caches add VRAM. A speedup is never claimed without measured results.

**Next step:** If all-8-bit quality passes but inference does not speed up, explore low-bit native runtime kernels or PyTorch `torchao` backends supported by the actual target GPU. If 4-bit profiles pass consistently, that is an opportunity for a genuinely low-bit optimized GEMM on an L4 or newer GPU.
